## Compound Flood Risk Analysis — Snowmelt (SWE) & Soil Moisture maps
CESM2-LE vs ERA5-interpolated, same methodology as in `create_precip_maps_hans.ipynb`
(2-day median + 90th-percentile + percentile significance testing + seasonal analysis + diagnostics for testing). Snowmelt = multi-day melt; soil moisture = multi-day rolling mean. Only 90 of the 100 CESM2-LE
members for SWE/soil-moisture, so ensemble size is auto-detected (n=90).
Output → `figures/compound_flood_risk_output/`.


### Time-Period Selector for the Whole Analysis

In [1]:
# %% [Setup, imports, per-variable configuration]
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.feature as cfeature  # used by plot_style internals
import dask
dask.config.set(scheduler="synchronous")

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg
from plot_style import (
    plot_2day_interp_3panel, plot_2day_interp_diffonly,
    plot_2day_interp_diffonly_sig, plot_2day_interp_seasonal_4row_3col,
)
from data_era5 import (
    compute_era5_interpolated_2day_median_2d,
    compute_era5_interpolated_2day_p90_2d,
    compute_era5_interpolated_2day_seasonal_median_2d,
    compute_era5_interpolated_2day_seasonal_p90_2d,
)
from data_smile import (
    compute_cesm2_le_2day_global_median_2d,
    compute_cesm2_le_2day_per_member_p90_2d,
    compute_significance_masks,
    compute_cesm2_le_2day_seasonal_global_median_2d,
    compute_cesm2_le_2day_seasonal_per_member_p90_2d,
)
from catchment_tools import (
    load_catchments, rolling_change, rolling_melt, rolling_mean,
    subset_time_series_by_year, open_field_cache,
)

# ── Output + analysis settings (mirror create_precip_maps_hans.ipynb) ─────────
FIG_SUBDIR     = "compound_flood_risk_output"
MAP_START      = 1995
MAP_END        = 2024
MAP_EXTENT_ANN = (5.0, 14.0, 57.5, 64.0)
RECOMPUTE      = True   # True → rebuild ALL caches (daily per-member + derived); slow

CATCHMENT_NUMBERS = {
    "nevina_bergheim":  1, "nevina_honnefoss": 2, "nevina_losna": 3,
    "regine_drammen":   4, "regine_glomma":    5,
}
CATCHMENT_LEGEND_TEXT = (
    "Catchments:\n  1 · Nevina Bergheim\n  2 · Nevina Hønnefoss\n  3 · Nevina Losna\n"
    "  4 · Regine Drammen\n  5 · Regine Glomma"
)
CATCHMENT_LABEL_OVERRIDES = {
    "nevina_bergheim": (8.3, 60.8), "regine_drammen": (9.8, 60.2),
}
SIG_LEGEND_5_95 = ("Significance (p=0.10, 2-sided):\n"
                   "  //  CESM2-LE > ERA5-Interp.\n  \\\\ ERA5-Interp. > CESM2-LE")
SIG_LEGEND_2_98 = ("Significance (p=0.04, 2-sided):\n"
                   "  //  CESM2-LE > ERA5-Interp.\n  \\\\ ERA5-Interp. > CESM2-LE")

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.titlesize": 12, "axes.labelsize": 10, "figure.dpi": 150})

# ── Dependency callables passed into helper functions ─────────────────────────
_sub           = subset_time_series_by_year
_roll_change   = rolling_change                      # 2-day signed SWE change (gain +, melt −)
_roll_melt     = rolling_melt                        # 2-day snowmelt = max(0, −ΔSWE), melt ≥ 0
_roll_mean     = rolling_mean                        # 2-day rolling mean (soil moisture)
def _open_field(kind):
    return lambda p, s, e: open_field_cache(p, kind, s, e)
def figp(fname):
    return cfg.precip_map_figure_paths(FIG_SUBDIR, fname)

# ── Per-variable configuration ────────────────────────────────────────────────
# we consider kg/m2 and snowmelt is generally close to zero for the median across one whole year (averages out)
# 
VARIABLES = {
        "snowmelt": dict(
        kind="swe", var_word="snowmelt", noun="Snowmelt",
        fstem="dailymedian",   # filename temporal prefix (snowmelt = 24-h ΔSWE)
        cesm2_dir=cfg.CESM2_LE_SWE_DIR, era5_interp_dir=cfg.ERA5_INTERPOLATED_SWE_DIR,
        cesm2_raw_var="SWE", era5_raw_var="sd",
        roll=_roll_change, open=_open_field("swe"),   # signed ΔSWE = SWE(t) − SWE(t−1), one value/date
        median_vmax=5.0, p90_vmax=20.0,
        seq_label_median="ΔSWE Median (kg/m²)",
        seq_label_p90="90th-pctl. ΔSWE (kg/m²)",
        div_label_median="Difference of ΔSWE Median (%),\nCESM2-LE \u2013 ERA5 Interpolated",
        div_label_p90="90th-pctl. ΔSWE Difference (%),\nCESM2-LE \u2013 ERA5 Interpolated",
        title_median="Snowmelt (ΔSWE) Median Difference",
        title_p90="90th-Percentile Snowmelt (ΔSWE) Difference",
        title_seasonal_median="Seasonal Snowmelt (ΔSWE) Median Difference",
        title_seasonal_p90="Seasonal 90th-Percentile Snowmelt (ΔSWE) Difference",
        diag_med_label="median ΔSWE = SWE(t)−SWE(t−1) (kg/m²)",
        diag_p90_label="90th pctl. ΔSWE = SWE(t)−SWE(t−1) (kg/m²)",
    ),
    "soil_moisture": dict(
        kind="soil_moisture", var_word="soil_moisture", noun="Soil Moisture",
        fstem="2daymedian",   # filename temporal prefix (soil moisture = 2-day mean)
        cesm2_dir=cfg.CESM2_LE_SM_DIR, era5_interp_dir=cfg.ERA5_INTERPOLATED_SWVL_DIR,
        cesm2_raw_var="SM", era5_raw_var="swvl",
        roll=_roll_mean, open=_open_field("soil_moisture"),
        median_vmax=3500.0, p90_vmax=3500.0,
        seq_label_median="Soil Moisture Median (kg/m²)",
        seq_label_p90="90th-pctl. Soil Moisture (kg/m²)",
        div_label_median="Difference of Soil Moisture Median (%),\nCESM2-LE \u2013 ERA5 Interpolated",
        div_label_p90="90th-pctl. Soil Moisture Difference (%),\nCESM2-LE \u2013 ERA5 Interpolated",
        title_median="Soil Moisture Median Difference",
        title_p90="90th-Percentile Soil Moisture Difference",
        title_seasonal_median="Seasonal Soil Moisture Median Difference",
        title_seasonal_p90="Seasonal 90th-Percentile Soil Moisture Difference",
        diag_med_label="daily SM median (kg/m²)",
        diag_p90_label="90th pctl. daily SM (kg/m²)",
    ),
}

print("Loading catchment polygons ...")
catchments_maps = load_catchments(cfg.GEOJSON_FILES, cfg.GEOJSON_DIR)
STORE, SEASONAL_MED, SEASONAL_P90 = {}, {}, {}
print("Setup done.")


Loading catchment polygons ...
Setup done.


In [2]:
# %% [One-time daily cache builder — run once; set RECOMPUTE=True (Cell 1) to rebuild]
# Loads the WHOLE available timeseries once and stores it as postprocessed daily
# caches (per-member CESM2-LE 1920–2034; ERA5-interp 1941–2025) under:
#   cesm2_le/swe, cesm2_le/soil_moisture, era5_interpolated/swe, era5_interpolated/soil_moisture
# Per-member caches keep single-member values. All later cells read these caches;
# derived caches (median/p90/seasonal) are built lazily and reused too (same flag).
from data_smile import save_cesm2_le_field_overall
from data_era5   import save_era5_interpolated_field_overall

for _vkey, _V in VARIABLES.items():
    k = _V["kind"]
    print(f"\n=== {_V['noun']} — per-member daily CESM2-LE caches ===")
    save_cesm2_le_field_overall(
        _V["cesm2_dir"],
        (lambda mid, s, e, k=k: cfg.field_daily_cache_path("cesm2_le", k, s, e, member_id=mid)),
        variable=_V["cesm2_raw_var"], cache_var=k, units="kg/m2", force=RECOMPUTE)
    print(f"\n=== {_V['noun']} — ERA5-interpolated daily cache ===")
    save_era5_interpolated_field_overall(
        _V["era5_interp_dir"],
        (lambda s, e, k=k: cfg.field_daily_cache_path("era5_interpolated", k, s, e)),
        variable=_V["era5_raw_var"], cache_var=k, units="kg/m2",
        extent=cfg.OVERALL_PRECIP_EXTENT, force=RECOMPUTE)

print("\nAll daily caches up to date.")


=== Snowmelt — per-member daily CESM2-LE caches ===
  cesm2_le/swe: 90 members, 1920–2034 ...
    [build] member 002 (1/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member002_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 004 (2/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member004_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 006 (3/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member006_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 008 (4/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member008_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 010 (5/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member010_1920-2034.nc  ({'ti

### Load once and store in post-processed folder SWE and Soil Moisture Data

In [3]:
# %% [Seasonal 2-day median + p90 computation — cache builder
for _vkey, _V in VARIABLES.items():
    print(f"\n{'='*60}\n{_V['noun']} — seasonal computation\n{'='*60}")
    k = _V["kind"]
    _md = (lambda mid, s, e, k=k: cfg.field_daily_cache_path("cesm2_le", k, s, e, member_id=mid))
    _ed = (lambda ds, res, s, e, k=k: cfg.field_daily_cache_path("era5_interpolated", k, s, e))

    SEASONAL_MED[_vkey], SEASONAL_P90[_vkey] = {}, {}

    for _s in cfg.SEASONS_ORDER:
        # ── seasonal median ──
        _pm_med_path = cfg.field_2day_cache_path("cesm2_le", k, "per_member_medians", MAP_START, MAP_END, season=_s,)
        da_c_med, _ = compute_cesm2_le_2day_seasonal_global_median_2d(
            _s, MAP_START, MAP_END, _V["cesm2_dir"], _md,
            cfg.field_2day_cache_path("cesm2_le", k, "global_median", MAP_START, MAP_END, season=_s,),
            _pm_med_path,
            _V["open"], _V["roll"], _sub, force_recompute=RECOMPUTE)
        da_e_med = compute_era5_interpolated_2day_seasonal_median_2d(
            _s, MAP_START, MAP_END, _V["era5_interp_dir"],
            cfg.field_2day_cache_path("era5_interpolated", k, "median", MAP_START, MAP_END, season=_s),
            _ed, _V["open"], _V["roll"], _sub, force_recompute=RECOMPUTE)
        _safe_med = da_e_med.where(da_e_med != 0)
        da_diff_med = (da_c_med - da_e_med) / _safe_med * 100.0
        SEASONAL_MED[_vkey][_s] = dict(per_member_path=_pm_med_path,
                                       da_cesm2=da_c_med, da_era5_interp=da_e_med,
                                       da_diff=da_diff_med)

        # ── seasonal 90th percentile ──
        _pm_p90_path = cfg.field_2day_cache_path("cesm2_le", k, "per_member_p90", MAP_START, MAP_END, season=_s)
        da_c_p90, _ = compute_cesm2_le_2day_seasonal_per_member_p90_2d(
            _s, MAP_START, MAP_END, _V["cesm2_dir"], _md,
            _pm_p90_path,
            cfg.field_2day_cache_path("cesm2_le", k, "global_p90", MAP_START, MAP_END, season=_s),
            _V["open"], _V["roll"], _sub, force_recompute=RECOMPUTE)
        da_e_p90 = compute_era5_interpolated_2day_seasonal_p90_2d(
            _s, MAP_START, MAP_END, _V["era5_interp_dir"],
            cfg.field_2day_cache_path("era5_interpolated", k, "p90", MAP_START, MAP_END, season=_s),
            _ed, _V["open"], _V["roll"], _sub, force_recompute=RECOMPUTE)
        _safe_p90 = da_e_p90.where(da_e_p90 != 0)
        da_diff_p90 = (da_c_p90 - da_e_p90) / _safe_p90 * 100.0
        SEASONAL_P90[_vkey][_s] = dict(per_member_path=_pm_p90_path,
                                       da_cesm2=da_c_p90, da_era5_interp=da_e_p90,
                                       da_diff=da_diff_p90)
print("\nDone (seasonal median + p90 computation).")



Snowmelt — seasonal computation
  Computing CESM2-LE seasonal DJF 90-member 2-day global median (1995–2024) ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keep

    10/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keep

/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


...

    70/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    80/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    90/90 members processed ...
  [saved] cesm2_2day_seasonal_SON_global_p90_soil_moisture_1995-2024.nc
  [saved] cesm2_2day_seasonal_SON_per_member_p90_soil_moisture_1995-2024.nc
  Computing ERA5-interp seasonal SON 2-day p90 (1995–2024) ...
  [saved] 2day_seasonal_SON_p90_era5interp_soil_moisture_1995-2024.nc

Done (seasonal median + p90 computation).


### Create 2-day median + 90th pctl. difference + diffonly plots for SWE and Soil Moisture

In [4]:
# %% [2-day median + 90th-pctile 3-panel diff + diffonly — both variables]
for _vkey, _V in VARIABLES.items():
    print(f"\n{'='*60}\n{_V['noun']}\n{'='*60}")
    k, vw, fstem = _V["kind"], _V["var_word"], _V["fstem"]
    _md = (lambda mid, s, e, k=k: cfg.field_daily_cache_path("cesm2_le", k, s, e, member_id=mid))
    _ed = (lambda ds, res, s, e, k=k: cfg.field_daily_cache_path("era5_interpolated", k, s, e))

    da_era5_med = compute_era5_interpolated_2day_median_2d(
        MAP_START, MAP_END, _V["era5_interp_dir"], _ed, _V["open"], _V["roll"], _sub)

    da_cesm2_med = compute_cesm2_le_2day_global_median_2d(
        MAP_START, MAP_END, _V["cesm2_dir"], _md,
        cfg.field_2day_cache_path("cesm2_le", k, "global_median", MAP_START, MAP_END),
        cfg.field_2day_cache_path("cesm2_le", k, "per_member_medians", MAP_START, MAP_END),
        _V["open"], _V["roll"], _sub, force_recompute=RECOMPUTE)

    da_cesm2_p90, _ = compute_cesm2_le_2day_per_member_p90_2d(
        MAP_START, MAP_END, _V["cesm2_dir"], _md,
        cfg.field_2day_cache_path("cesm2_le", k, "per_member_p90", MAP_START, MAP_END),
        cfg.field_2day_cache_path("cesm2_le", k, "global_p90", MAP_START, MAP_END),
        _V["open"], _V["roll"], _sub, force_recompute=RECOMPUTE)

    da_era5_p90 = compute_era5_interpolated_2day_p90_2d(
        MAP_START, MAP_END, _V["era5_interp_dir"],
        cfg.field_2day_cache_path("era5_interpolated", k, "p90", MAP_START, MAP_END),
        _ed, _V["open"], _V["roll"], _sub, force_recompute=RECOMPUTE)

    _safe_med = da_era5_med.where(da_era5_med != 0)
    da_diff_med = (da_cesm2_med - da_era5_med) / _safe_med * 100.0
    _safe_p90 = da_era5_p90.where(da_era5_p90 != 0)
    da_diff_p90 = (da_cesm2_p90 - da_era5_p90) / _safe_p90 * 100.0

    STORE[_vkey] = dict(cesm2_med=da_cesm2_med, era5_med=da_era5_med, diff_med=da_diff_med,
                        cesm2_p90=da_cesm2_p90, era5_p90=da_era5_p90, diff_p90=da_diff_p90)

    common = dict(catchments=catchments_maps, start_year=MAP_START, end_year=MAP_END,
                  catchment_numbers=CATCHMENT_NUMBERS, catchment_legend_text=CATCHMENT_LEGEND_TEXT,
                  label_overrides=CATCHMENT_LABEL_OVERRIDES, annmedian_extent=MAP_EXTENT_ANN)

    plot_2day_interp_3panel(
        da_cesm2=da_cesm2_med, da_era5_interp=da_era5_med, da_diff=da_diff_med,
        out_paths=figp(f"{fstem}_{vw}_diff_{MAP_START}-{MAP_END}.pdf"),
        fig_title=f"{_V['title_median']} ({MAP_START}–{MAP_END})",
        seq_cbar_label=_V["seq_label_median"], div_cbar_label=_V["div_label_median"],
        twoday_vmax=_V["median_vmax"], **common)
    plot_2day_interp_diffonly(
        da_diff=da_diff_med,
        out_paths=figp(f"{fstem}_{vw}_diffonly_{MAP_START}-{MAP_END}.pdf"),
        fig_title=f"{_V['title_median']} ({MAP_START}–{MAP_END})",
        div_cbar_label=_V["div_label_median"], **common)
    plot_2day_interp_3panel(
        da_cesm2=da_cesm2_p90, da_era5_interp=da_era5_p90, da_diff=da_diff_p90,
        out_paths=figp(f"{fstem}_90pctl_{vw}_diff_{MAP_START}-{MAP_END}.pdf"),
        fig_title=f"{_V['title_p90']} ({MAP_START}–{MAP_END})",
        seq_cbar_label=_V["seq_label_p90"], div_cbar_label=_V["div_label_p90"],
        twoday_vmax=_V["p90_vmax"], **common)
    plot_2day_interp_diffonly(
        da_diff=da_diff_p90,
        out_paths=figp(f"{fstem}_90pctl_{vw}_diffonly_{MAP_START}-{MAP_END}.pdf"),
        fig_title=f"{_V['title_p90']} ({MAP_START}–{MAP_END})",
        div_cbar_label=_V["div_label_p90"], **common)
print("\nDone (base diff + diffonly figures).")



Snowmelt
  Loading ERA5-interpolated (2-day median) from cache (1995–2024) ...
  Computing CESM2-LE 90-member 2-day GLOBAL median (1995–2024) — this reads all members ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keep

    10/90 members loaded ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keep

    20/90 members loaded ...


...

    70/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    80/90 members processed ...


/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/nird/home/lbal/venvs/storm_hans/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a

    90/90 members processed ...
  [saved] cesm2_2day_global_p90_soil_moisture_1995-2024.nc
  [saved] cesm2_2day_per_member_p90_soil_moisture_1995-2024.nc
  Computing ERA5-interpolated 2-day p90 (1995–2024) ...
  [saved] 2day_p90_era5interp_soil_moisture_1995-2024.nc
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_soil_moisture_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_soil_moisture_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_soil_moisture_diffonly_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_soil_moisture_diffonly_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/2daymedian_90pctl_soil_moisture_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/2daymedian_90pctl_soil_moisture_diff_1995-

### Create 2-day median + 90th pctl. difference + diffonly plots with Significance hatching! for SWE and Soil Moisture

In [5]:
# %% [Significance-hatched 2-day median + p90 diff — 5/95 and 2/98 — both variables]
for _vkey, _V in VARIABLES.items():
    print(f"\n{'='*60}\n{_V['noun']} — significance\n{'='*60}")
    S, k, vw, fstem = STORE[_vkey], _V["kind"], _V["var_word"], _V["fstem"]

    with xr.open_dataset(str(cfg.field_2day_cache_path("cesm2_le", k, "per_member_medians", MAP_START, MAP_END))) as _ds:
        pm_med = _ds[list(_ds.data_vars)[0]].load()
    with xr.open_dataset(str(cfg.field_2day_cache_path("cesm2_le", k, "per_member_p90", MAP_START, MAP_END))) as _ds:
        pm_p90 = _ds[list(_ds.data_vars)[0]].load()

    common = dict(catchments=catchments_maps, start_year=MAP_START, end_year=MAP_END,
                  catchment_numbers=CATCHMENT_NUMBERS, catchment_legend_text=CATCHMENT_LEGEND_TEXT,
                  label_overrides=CATCHMENT_LABEL_OVERRIDES, annmedian_extent=MAP_EXTENT_ANN)

    for lo, hi, tag, sig_leg in [(5.0, 95.0, "5_95pctl", SIG_LEGEND_5_95),
                                 (2.0, 98.0, "2_98pctl", SIG_LEGEND_2_98)]:
        sc, se = compute_significance_masks(S["era5_med"], pm_med, lower_pctl=lo, upper_pctl=hi)
        plot_2day_interp_3panel(
            da_cesm2=S["cesm2_med"], da_era5_interp=S["era5_med"], da_diff=S["diff_med"],
            out_paths=figp(f"{fstem}_{vw}_{tag}_diff_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{_V['title_median']} ({MAP_START}–{MAP_END})",
            seq_cbar_label=_V["seq_label_median"], div_cbar_label=_V["div_label_median"],
            twoday_vmax=_V["median_vmax"], sig_cesm_higher=sc, sig_era5_higher=se,
            sig_legend_text=sig_leg, **common)
        plot_2day_interp_diffonly_sig(
            da_diff=S["diff_med"],
            out_paths=figp(f"{fstem}_{vw}_{tag}_diffonly_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{_V['title_median']} ({MAP_START}–{MAP_END})",
            div_cbar_label=_V["div_label_median"], sig_cesm_higher=sc, sig_era5_higher=se,
            sig_legend_text=sig_leg, **common)
        sc, se = compute_significance_masks(S["era5_p90"], pm_p90, lower_pctl=lo, upper_pctl=hi)
        plot_2day_interp_3panel(
            da_cesm2=S["cesm2_p90"], da_era5_interp=S["era5_p90"], da_diff=S["diff_p90"],
            out_paths=figp(f"{fstem}_90pctl_{vw}_{tag}_diff_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{_V['title_p90']} ({MAP_START}–{MAP_END})",
            seq_cbar_label=_V["seq_label_p90"], div_cbar_label=_V["div_label_p90"],
            twoday_vmax=_V["p90_vmax"], sig_cesm_higher=sc, sig_era5_higher=se,
            sig_legend_text=sig_leg, **common)
        plot_2day_interp_diffonly_sig(
            da_diff=S["diff_p90"],
            out_paths=figp(f"{fstem}_90pctl_{vw}_{tag}_diffonly_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{_V['title_p90']} ({MAP_START}–{MAP_END})",
            div_cbar_label=_V["div_label_p90"], sig_cesm_higher=sc, sig_era5_higher=se,
            sig_legend_text=sig_leg, **common)
print("\nDone (significance figures).")



Snowmelt — significance
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/dailymedian_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/dailymedian_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/dailymedian_snowmelt_5_95pctl_diffonly_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/dailymedian_snowmelt_5_95pctl_diffonly_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/dailymedian_90pctl_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/dailymedian_90pctl_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/dailymedian_90pctl_snowmelt_5_95pctl_diffonly_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_ris

### Diagnostic Tables for Pixels: Overview over Values

In [6]:
# %% [Diagnostic: pixel-level tables — CESM2-LE vs ERA5-interp (median & p90)]
for _vkey, _V in VARIABLES.items():
    S = STORE[_vkey]
    for _stat, _c, _e, _d in [("median", S["cesm2_med"], S["era5_med"], S["diff_med"]),
                              ("p90",    S["cesm2_p90"], S["era5_p90"], S["diff_p90"])]:
        _lats, _lons = _e["lat"].values, _e["lon"].values
        _cv = _c.sel(lat=_lats, lon=_lons).values
        _ev = _e.values
        _dv = _d.sel(lat=_lats, lon=_lons).values
        _rows = [{"lat": round(float(la), 3), "lon": round(float(lo), 3),
                  "cesm2": round(float(_cv[i, j]), 3), "era5": round(float(_ev[i, j]), 3),
                  "diff_pct": round(float(_dv[i, j]), 1)}
                 for i, la in enumerate(_lats) for j, lo in enumerate(_lons)]
        print(f"\n=== {_V['noun']} — {_stat} (kg/m²) ===")
        print(pd.DataFrame(_rows).to_string(index=False, na_rep="NaN"))



=== Snowmelt — median (kg/m²) ===
   lat   lon  cesm2  era5  diff_pct
56.073  2.50    NaN   0.0       NaN
56.073  3.75    NaN   0.0       NaN
56.073  5.00    NaN   0.0       NaN
56.073  6.25    NaN   0.0       NaN
56.073  7.50    NaN   0.0       NaN
56.073  8.75    0.0   0.0       NaN
56.073 10.00    0.0   0.0       NaN
56.073 11.25    0.0   0.0       NaN
56.073 12.50    0.0   0.0       NaN
56.073 13.75    0.0   0.0       NaN
56.073 15.00    0.0   0.0       NaN
56.073 16.25    0.0   0.0       NaN
57.016  2.50    NaN   0.0       NaN
57.016  3.75    NaN   0.0       NaN
57.016  5.00    NaN   0.0       NaN
57.016  6.25    NaN   0.0       NaN
57.016  7.50    0.0   0.0       NaN
57.016  8.75    0.0   0.0       NaN
57.016 10.00    0.0   0.0       NaN
57.016 11.25    0.0   0.0       NaN
57.016 12.50    0.0   0.0       NaN
57.016 13.75    0.0   0.0       NaN
57.016 15.00    0.0   0.0       NaN
57.016 16.25    0.0   0.0       NaN
57.958  2.50    NaN   0.0       NaN
57.958  3.75    NaN   0.0    

### Seasonal Data Computation for SWE and Soil Moisture

In [7]:
# %% [Seasonal significance-hatched 4×3 plots — median & p90, 5/95 & 2/98 — both variables]
def _build_seasonal_list(store_for_var, lo, hi):
    out = []
    for _s in cfg.SEASONS_ORDER:
        d = store_for_var[_s]
        with xr.open_dataset(str(d["per_member_path"])) as _ds:
            pm = _ds[list(_ds.data_vars)[0]].load()
        sc, se = compute_significance_masks(d["da_era5_interp"], pm, lower_pctl=lo, upper_pctl=hi)
        out.append({"season_label": cfg.SEASON_LABELS[_s],
                    "da_cesm2": d["da_cesm2"], "da_era5_interp": d["da_era5_interp"],
                    "da_diff": d["da_diff"], "sig_cesm_higher": sc, "sig_era5_higher": se})
    return out

for _vkey, _V in VARIABLES.items():
    print(f"\n{_V['noun']} — seasonal significance plots ...")
    vw, fstem = _V["var_word"], _V["fstem"]
    common = dict(catchments=catchments_maps, start_year=MAP_START, end_year=MAP_END,
                  catchment_numbers=CATCHMENT_NUMBERS, catchment_legend_text=CATCHMENT_LEGEND_TEXT,
                  label_overrides=CATCHMENT_LABEL_OVERRIDES, annmedian_extent=MAP_EXTENT_ANN)
    for lo, hi, tag, sig_leg in [(5.0, 95.0, "5_95pctl", SIG_LEGEND_5_95),
                                 (2.0, 98.0, "2_98pctl", SIG_LEGEND_2_98)]:
        plot_2day_interp_seasonal_4row_3col(
            seasonal_data=_build_seasonal_list(SEASONAL_MED[_vkey], lo, hi),
            out_paths=figp(f"{fstem}_seasonal_{vw}_{tag}_diff_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{_V['title_seasonal_median']} ({MAP_START}–{MAP_END})",
            seq_cbar_label=_V["seq_label_median"], div_cbar_label=_V["div_label_median"],
            twoday_vmax=_V["median_vmax"], sig_legend_text=sig_leg, **common)
        plot_2day_interp_seasonal_4row_3col(
            seasonal_data=_build_seasonal_list(SEASONAL_P90[_vkey], lo, hi),
            out_paths=figp(f"{fstem}_seasonal_90pctl_{vw}_{tag}_diff_{MAP_START}-{MAP_END}.pdf"),
            fig_title=f"{_V['title_seasonal_p90']} ({MAP_START}–{MAP_END})",
            seq_cbar_label=_V["seq_label_p90"], div_cbar_label=_V["div_label_p90"],
            twoday_vmax=_V["p90_vmax"], sig_legend_text=sig_leg, **common)
print("\nDone (seasonal significance figures).")



Snowmelt — seasonal significance plots ...
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/dailymedian_seasonal_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/dailymedian_seasonal_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/dailymedian_seasonal_90pctl_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/dailymedian_seasonal_90pctl_snowmelt_5_95pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/dailymedian_seasonal_snowmelt_2_98pctl_diff_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/dailymedian_seasonal_snowmelt_2_98pctl_diff_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/dailymedian_seasonal_90pctl_snowmelt_2_98pctl_diff_1995-2024.pdf
Sa

### Diagnostic Tables for Pixels: Overview over Values

In [8]:
# %% [Seasonal diagnostic: per-season max values for colorbar calibration]
for _vkey, _V in VARIABLES.items():
    print("=" * 72)
    print(f"{_V['noun'].upper()} — SEASONAL MAX VALUES (kg/m²) FOR COLORBAR CALIBRATION")
    print("=" * 72)
    for _name, _store, _vmax in [("median", SEASONAL_MED[_vkey], _V["median_vmax"]),
                                 ("p90",    SEASONAL_P90[_vkey], _V["p90_vmax"])]:
        print(f"\n── {_name} (current vmax used: {_vmax}) ──")
        _allc = []
        for _s in cfg.SEASONS_ORDER:
            d = _store[_s]
            cmax = float(np.nanmax(d["da_cesm2"].values))
            emax = float(np.nanmax(d["da_era5_interp"].values))
            _allc.append(max(cmax, emax))
            print(f"  {cfg.SEASON_LABELS[_s]:<20} CESM2={cmax:>10.3f}  ERA5={emax:>10.3f}")
        print(f"  → suggested vmax: {max(_allc):.1f}")


SNOWMELT — SEASONAL MAX VALUES (kg/m²) FOR COLORBAR CALIBRATION

── median (current vmax used: 5.0) ──
  Winter (DJF)         CESM2=     4.123  ERA5=     1.701
  Spring (MAM)         CESM2=     0.651  ERA5=     0.000
  Summer (JJA)         CESM2=     0.000  ERA5=     0.000
  Autumn (SON)         CESM2=     0.001  ERA5=     0.000
  → suggested vmax: 4.1

── p90 (current vmax used: 20.0) ──
  Winter (DJF)         CESM2=    14.538  ERA5=     9.535
  Spring (MAM)         CESM2=     8.523  ERA5=     5.068
  Summer (JJA)         CESM2=     0.000  ERA5=     0.419
  Autumn (SON)         CESM2=     5.289  ERA5=     5.421
  → suggested vmax: 14.5
SOIL MOISTURE — SEASONAL MAX VALUES (kg/m²) FOR COLORBAR CALIBRATION

── median (current vmax used: 3500.0) ──
  Winter (DJF)         CESM2=  3219.635  ERA5=  1241.912
  Spring (MAM)         CESM2=  3272.274  ERA5=  1205.364
  Summer (JJA)         CESM2=  3151.340  ERA5=  1163.814
  Autumn (SON)         CESM2=  3124.960  ERA5=  1219.863
  → suggested vm

### Diagnostics for pixel-cell number 3

In [9]:
# %% [Diagnostic: catchment-3 (Losna) grid-cell strip plots — seasonal median & p90]
import geopandas as gpd
_C_CESM2, _C_ERA5 = "#2C7BB6", "#D73027"

_losna_gdf = gpd.read_file(cfg.GEOJSON_DIR / cfg.GEOJSON_FILES["nevina_losna"])
try:
    _pt = _losna_gdf.geometry.union_all().representative_point()
except AttributeError:
    _pt = _losna_gdf.geometry.unary_union.representative_point()
_sel_lon, _sel_lat = _pt.x, _pt.y

for _vkey, _V in VARIABLES.items():
    vw = _V["var_word"]
    for _which, _store, _ylabel, _fname in [
        ("median", SEASONAL_MED[_vkey], _V["diag_med_label"],
        f"diagnostic_catchment3_2daymedian_seasonal_{vw}_{MAP_START}-{MAP_END}.pdf"),
        ("p90", SEASONAL_P90[_vkey], _V["diag_p90_label"],
        f"diagnostic_catchment3_2day90pctl_seasonal_{vw}_{MAP_START}-{MAP_END}.pdf"),
    ]:
        _fig, _axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
        _fig.suptitle(f"CESM2-LE ({_V['noun']}) vs ERA5 Interpolated — Catchment 3 (Losna) grid cell\n"
                      f"Seasonal {_which}  ({MAP_START}–{MAP_END})", fontsize=13, y=1.03)
        for _ax, _season in zip(_axes, cfg.SEASONS_ORDER):
            with xr.open_dataset(str(_store[_season]["per_member_path"])) as _ds:
                _pm = _ds[list(_ds.data_vars)[0]]
                _vals = _pm.sel(lat=_sel_lat, lon=_sel_lon, method="nearest").values.ravel()
            _era5v = float(_store[_season]["da_era5_interp"].sel(lat=_sel_lat, lon=_sel_lon, method="nearest"))
            _n = len(_vals)
            _ax.scatter(np.arange(1, _n + 1), _vals, color=_C_CESM2, s=14, alpha=0.65, zorder=2,
                        label=f"CESM2-LE ({_n} members)")
            _ax.scatter(0, _era5v, color=_C_ERA5, s=100, marker="D", zorder=4,
                        label=f"ERA5 Interp. ({_era5v:.2f})")
            _ax.axvline(0.5, color="0.75", lw=0.8, ls="--", zorder=1)
            _ax.set_title(cfg.SEASON_LABELS[_season], fontsize=11)
            _ax.set_xlabel("Member index", fontsize=9); _ax.set_xlim(-3, _n + 2)
            _ax.spines["top"].set_visible(False); _ax.spines["right"].set_visible(False)
        _axes[0].set_ylabel(_ylabel, fontsize=10)
        _axes[-1].legend(fontsize=9, frameon=False, loc="upper right")
        _fig.tight_layout()
        for _root in (cfg.FIGURES_DIR, cfg.FIGURES_DIR_SECONDARY):
            _p = _root / FIG_SUBDIR / _fname
            _p.parent.mkdir(parents=True, exist_ok=True)
            _fig.savefig(str(_p), bbox_inches="tight", dpi=150)
            print(f"Saved → {_p}")
        plt.close(_fig)


Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/diagnostic_catchment3_2daymedian_seasonal_snowmelt_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/diagnostic_catchment3_2daymedian_seasonal_snowmelt_1995-2024.pdf


Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/diagnostic_catchment3_2day90pctl_seasonal_snowmelt_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/diagnostic_catchment3_2day90pctl_seasonal_snowmelt_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/diagnostic_catchment3_2daymedian_seasonal_soil_moisture_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/diagnostic_catchment3_2daymedian_seasonal_soil_moisture_1995-2024.pdf
Saved → /nird/datalake/NS9873K/lbal/figures/compound_flood_risk_output/diagnostic_catchment3_2day90pctl_seasonal_soil_moisture_1995-2024.pdf
Saved → /nird/home/lbal/internship_storm_hans/figures/compound_flood_risk_output/diagnostic_catchment3_2day90pctl_seasonal_soil_moisture_1995-2024.pdf
